# Part 2 — Model Selection

**Selected Model:** Qwen 2.5 7B Instruct  
**HuggingFace ID:** `Qwen/Qwen2.5-7B-Instruct`  
**Quantization:** 4-bit NF4 (bitsandbytes) — required for 8-12GB VRAM

This notebook covers:
- Hardware / environment description
- Model selection justification
- Model loading
- Sanity check with sample translations

## 2.1 Experimental Environment

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import platform
import subprocess

from modules.model import gpu_info

In [ ]:
info = gpu_info()

print("=" * 50)
print("EXPERIMENTAL ENVIRONMENT")
print("=" * 50)
print(f"  OS              : {platform.system()} {platform.release()}")
print(f"  Python          : {sys.version.split()[0]}")
print(f"  PyTorch         : {torch.__version__}")
print(f"  CUDA available  : {torch.cuda.is_available()}")

if info['available']:
    print(f"  GPU Name        : {info['name']}")
    print(f"  VRAM (total)    : {info['total_vram_gb']} GB")
    print(f"  CUDA version    : {info['cuda_version']}")
    
    # Current VRAM usage
    free, total = torch.cuda.mem_get_info()
    print(f"  VRAM (free now) : {free/1e9:.1f} GB")
else:
    print("  GPU             : NOT AVAILABLE (CPU mode)")

# RAM
try:
    result = subprocess.run(['free', '-h'], capture_output=True, text=True)
    for line in result.stdout.splitlines():
        if 'Mem' in line:
            parts = line.split()
            print(f"  RAM (total)     : {parts[1]}")
            print(f"  RAM (free)      : {parts[3]}")
except Exception:
    pass

print("=" * 50)

## 2.2 Model Selection Justification

### Why Qwen 2.5 7B Instruct?

| Criterion | Justification |
|---|---|
| **Hardware fit** | 4-bit NF4 quantization reduces memory footprint to ~4-5GB VRAM, fitting comfortably in 8-12GB |
| **Multilingual support** | Qwen 2.5 was trained on data covering 29+ languages including Turkish |
| **Instruction-following** | Instruction-tuned variant reliably follows structured multi-step prompts (required for paper strategy) |
| **Translation quality** | Strong performance on multilingual benchmarks; outperforms Mistral 7B on non-European languages |
| **Context length** | 128K token context window — accommodates long few-shot RAG prompts |
| **Open weights** | Freely available on HuggingFace, no API key required |
| **Chat template** | Official chat template via `apply_chat_template` ensures correct formatting |

### Software Libraries

| Library | Version | Purpose |
|---|---|---|
| `transformers` | ≥4.40 | Model loading, tokenization, generation |
| `bitsandbytes` | ≥0.43 | 4-bit NF4 quantization |
| `accelerate` | ≥0.27 | Device map / multi-GPU support |
| `datasets` | ≥2.18 | WMT16 dataset loading |
| `sentence-transformers` | ≥2.7 | Multilingual embeddings for RAG |
| `faiss-cpu` | ≥1.7.4 | Vector similarity search |
| `unbabel-comet` | ≥2.2.1 | COMET MT evaluation metric |

## 2.3 Load Model

In [ ]:
from modules.model import load_model, generate

# quantize=True → 4-bit NF4 (recommended for 8-12GB VRAM)
model, tokenizer = load_model(quantize=True)

In [ ]:
# Memory usage after loading
if torch.cuda.is_available():
    allocated = torch.cuda.memory_allocated() / 1e9
    reserved  = torch.cuda.memory_reserved() / 1e9
    print(f"VRAM after model load:")
    print(f"  Allocated : {allocated:.2f} GB")
    print(f"  Reserved  : {reserved:.2f} GB")

## 2.4 Sanity Check — Sample Translations

In [ ]:
from modules.prompts import zero_shot_prompt

test_sentences = [
    "The European Parliament voted in favor of the new climate policy.",
    "Scientists have discovered a new species of fish in the deep ocean.",
    "The prime minister announced a series of economic reforms aimed at reducing unemployment.",
]

print("Sanity check — Zero-shot EN→TR translations:\n")
for i, sent in enumerate(test_sentences, 1):
    prompt = zero_shot_prompt(sent, direction='en->tr')
    output = generate(model, tokenizer, prompt)
    print(f"[{i}] EN: {sent}")
    print(f"    TR: {output}")
    print()

In [ ]:
# Save model and tokenizer references for downstream notebooks
# (re-load in each notebook to avoid memory issues between sessions)
print("Model is ready. Load it in downstream notebooks with:")
print("  from modules.model import load_model")
print("  model, tokenizer = load_model(quantize=True)")